# sentinel_fusion_ai — Model Training

Pipeline: Kaggle download → extract → fastai train → evaluate → export `.pkl` → benchmark.

## 1. Load Kaggle API token from .env

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

username = os.getenv("KAGGLE_USERNAME")
key = os.getenv("KAGGLE_TOKEN")

print(username)
print("Key loaded:", key is not None)

In [ ]:
import os

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_TOKEN"] = key

## 2. Find and download dataset

Set `DATASET_SLUG` to the Kaggle dataset you want (owner/dataset-name).

In [ ]:
!kaggle datasets list -s "YOUR SEARCH TERM"

In [ ]:
DATASET_SLUG = "owner/dataset-name"   # <-- CHANGE ME

!kaggle datasets download -d {DATASET_SLUG} -p data/

In [ ]:
import zipfile
from pathlib import Path

zip_path = next(Path("data").glob("*.zip"))
extract_path = Path("data/raw")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully to", extract_path)

In [ ]:
from pathlib import Path

path = Path("data/raw")

for item in path.iterdir():
    print(item)

## 3. Create model and train

In [ ]:
from fastai.vision.all import *
from pathlib import Path

# Fix dark mode rendering of fastprogress tables
try:
    from IPython.display import HTML, display
    display(HTML('''
    <style>
        table.fastprogress, .fastprogress td, .fastprogress th { color: inherit !important; }
    </style>
    '''))
except Exception:
    pass

In [ ]:
path = Path("data/raw")

# If dataset ships pre-split Training/Testing folders, point at them here:
# train_path = path / "Training"
# test_path  = path / "Testing"

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    # Pre-split dataset: use GrandparentSplitter(train_name="Training", valid_name="Testing")
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms=Resize(224),
).dataloaders(path, bs=32)

In [ ]:
dls.show_batch(max_n=9, figsize=(8,8))
dls.vocab

In [ ]:
learn = vision_learner(
    dls,
    resnet18,
    metrics=accuracy
)

learn.fine_tune(5)

## 4. Evaluate

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(6,6))

In [ ]:
# Predict a single image
# img = PILImage.create("data/raw/<class>/<some_image>.jpg")
# pred, pred_idx, probs = learn.predict(img)
# print(f"This is a: {pred}")
# for c, p in zip(dls.vocab, probs):
#     print(f"{c:20s}: {p:.4f}")

## 5. Export model

In [ ]:
MODEL_NAME = "sentinel_fusion_model.pkl"

learn.export(MODEL_NAME)

In [ ]:
from pathlib import Path

model_path = Path(MODEL_NAME)
size_mb = model_path.stat().st_size / (1024 * 1024)
print(f"Model Size: {size_mb:.2f} MB")

## 6. Benchmark

In [ ]:
total_params = sum(p.numel() for p in learn.model.parameters())
trainable_params = sum(p.numel() for p in learn.model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

In [ ]:
params = sum(p.numel() for p in learn.model.parameters())
ram = params * 4 / (1024**2)
print(f"Weights occupy approximately {ram:.2f} MB")

In [ ]:
import torch, time

# img = PILImage.create("data/raw/<class>/<some_image>.jpg")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    # learn.predict(img)
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1024**2:.2f} MB")

# start = time.time()
# learn.predict(img)
# print(f"Inference latency: {(time.time()-start)*1000:.2f} ms")

## 7. Test exported model

In [ ]:
from fastai.vision.all import *
from pathlib import Path

learn_inf = load_learner(MODEL_NAME)
categories = learn_inf.dls.vocab
print(categories)

In [ ]:
test_path = Path("data/raw")   # point at held-out test folder if available

correct = 0
total = 0

for cat in categories:
    img_files = get_image_files(test_path/cat)[:10]
    for img in img_files:
        pred, _, _ = learn_inf.predict(img)
        if pred == cat:
            correct += 1
        total += 1

print(f"Accuracy on sample: {correct}/{total} = {correct/total:.2%}")